# Cleaning data
In dit document gaan we alle features schoonmaken en een nieuwe dataset aanmaken waar all die features opgeschoont zijn.   

Dit zijn de features die we gaan opschonen:    
stm_sap_meld_ddt: Tijd van melding, continu.    
stm_geo_mld: Geolocatie, nominaal.    
stm_prioriteit: Prioriteitsindex, ordinaal.    
stm_aanngeb_dd: Datum aannemer gebeld, continu.    
stm_oorz_groep: Oorzaak groep storing, nominaal.   
stm_oorz_code: Oorzaak code storing, nominaal.    
stm_contractgeb_gst: Contractgebied aannnemer, nominaal.   
stm_techn_gst: Techniekveld melding, nominaal.   
stm_progfh_in_duur: Prognose aannemer duur functiehersteltijd, continu.   
stm_fh_status: De status van de functieherstel, ordinaal.  
stm_fh_tijd'
    ]

In [1]:
import numpy as np
import pandas as pd

In [2]:
file = pd.read_csv("sap_storing_data_hu_project.csv", low_memory=False)

## Duplicaten verwijderen en dataframe aanmaken
Eerst kijken we of er duplicates in de dataset zitten

In [3]:
file["#stm_sap_meldnr"].duplicated().sum()

332045

Hier zien we dat er inderdaad duplicaten in de dataset zitten, deze gaan we verwijderen

In [4]:
file = file.drop_duplicates(subset=["#stm_sap_meldnr"])

Nu maken we een dataframe aan met alle kolommen die we gaan schoonmaken

In [5]:
df = file[['stm_sap_meld_ddt', 'stm_geo_mld',  'stm_prioriteit', 'stm_aanngeb_dd', 'stm_oorz_groep', 'stm_oorz_code', 'stm_contractgeb_gst', 'stm_techn_gst', 'stm_progfh_in_duur', 'stm_fh_status']].copy()
df.columns

Index(['stm_sap_meld_ddt', 'stm_geo_mld', 'stm_prioriteit', 'stm_aanngeb_dd',
       'stm_oorz_groep', 'stm_oorz_code', 'stm_contractgeb_gst',
       'stm_techn_gst', 'stm_progfh_in_duur', 'stm_fh_status'],
      dtype='object')

Nu gaan we de kolommen in deze dataframe 1 voor 1 schoonmaken.

## stm_sap_meld_ddt
Eerst verwijderen we de NaN

In [6]:
df = df.dropna(subset=['stm_sap_meld_ddt'])
df['stm_sap_meld_ddt']

1         02/01/2006 09:00:00
2         02/01/2006 12:35:00
3         02/01/2006 16:40:00
4         02/01/2006 22:30:00
5         02/01/2006 11:23:00
                 ...         
898516    11/05/2013 07:55:00
898518    11/05/2013 07:59:00
898520    11/05/2013 08:06:00
898522    11/05/2013 09:21:00
898524    20/08/2016 14:15:17
Name: stm_sap_meld_ddt, Length: 566480, dtype: object

In deze kolom staan datums samen met de tijden van van de dag. Hier kunnen we heel makkelijk verkeerde values eruit halen met to_datetime en errors='coerce'. Dit zet namelijk alle verkeerde tijden om naar een NaN, deze kunnen we daarna heel makelijk verwijderen

In [7]:
df['stm_sap_meld_ddt'] = pd.to_datetime(df['stm_sap_meld_ddt'], errors='coerce')

df = df.dropna(subset=['stm_sap_meld_ddt'])
df['stm_sap_meld_ddt']

1        2006-02-01 09:00:00
2        2006-02-01 12:35:00
3        2006-02-01 16:40:00
4        2006-02-01 22:30:00
5        2006-02-01 11:23:00
                 ...        
898510   2013-11-05 07:17:00
898516   2013-11-05 07:55:00
898518   2013-11-05 07:59:00
898520   2013-11-05 08:06:00
898522   2013-11-05 09:21:00
Name: stm_sap_meld_ddt, Length: 225165, dtype: datetime64[ns]

Nu hebben we alle verkeerde values eruit gehaald, dit waren er zo te zien een hoop. Nu moeten we de tijden nog omzetten naar een getal die in model in kan. Dit doen met met de Unix-tijdstempel (1970-01-01 00:00:00). Om dit te doen moeten we van datetime64[ns] naar int gaan en dat delen voor 10**9. Uiteindelijk is dat getal het aantal seconden sinds 1970-01-01 00:00:00.

In [8]:
df['stm_sap_meld_ddt'] = df['stm_sap_meld_ddt'].astype('int64') 
df['stm_sap_meld_ddt'] = df['stm_sap_meld_ddt'] / (10**9)

In [9]:
df['stm_sap_meld_ddt'].sort_values()

215640    1.136074e+09
215645    1.136074e+09
215643    1.136077e+09
215641    1.136083e+09
215647    1.136085e+09
              ...     
894758    1.575672e+09
214168    1.575675e+09
214167    1.575675e+09
214173    1.575675e+09
139194    1.575677e+09
Name: stm_sap_meld_ddt, Length: 225165, dtype: float64

Nu hebben we er int van gemaakt die het aantal seconden sinds 1970-01-01 00:00:00 weergeeft

## stm_geo_mld
Eerst verwijderen we de NaN

In [10]:
df = df.dropna(subset=['stm_geo_mld'])
df['stm_geo_mld']

1         624.0
2         201.0
3          25.0
4          12.0
5         614.0
          ...  
898510    155.0
898516    118.0
898518    158.0
898520    560.0
898522    468.0
Name: stm_geo_mld, Length: 222317, dtype: object

Eerst kijken we naar alle unieke waardes.

In [11]:
print(sorted(set(df["stm_geo_mld"])))

['0', '001', '002', '004', '005', '006', '007', '008', '009', '011', '012', '013', '015', '017', '018', '019', '020', '021', '023', '024', '025', '026', '027', '028', '030', '031', '033', '034', '035', '036', '037', '038', '039', '040', '041', '042', '043', '044', '046', '047', '049', '050', '051', '052', '053', '054', '055', '056', '057', '058', '059', '060', '062', '063', '064', '065', '067', '070', '071', '073', '074', '075', '076', '078', '079', '080', '082', '083', '084', '085', '087', '088', '089', '090', '091', '092', '093', '094', '095', '096', '097', '098', '099', '1', '1.0', '10', '10.0', '100', '100.0', '101', '101.0', '102', '102.0', '103', '103.0', '104', '104.0', '105', '105.0', '106', '106.0', '107', '107.0', '108', '108.0', '109', '109.0', '11', '11.0', '110', '110.0', '111', '111.0', '112', '112.0', '114', '114.0', '115', '115.0', '116', '116.0', '117', '117.0', '118', '118.0', '119', '119.0', '12', '12.0', '120', '120.0', '121', '121.0', '122', '122.0', '123', '123.0'

Zo te zien staan er int en floats in de dataset als geocodes, deze horen hetzelfde te zijn, dus deze gaan we hetzelfde maken door ze hetzelfde datatype te maken.

In [12]:
df['stm_geo_mld'] = df['stm_geo_mld'].astype(float).astype(int)
print(sorted(set(df["stm_geo_mld"])))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 143, 144, 145, 146, 149, 152, 155, 158, 161, 163, 164, 165, 166, 200, 201, 203, 204, 205, 206, 208, 209, 210, 211, 212, 213, 216, 217, 223, 226, 228, 305, 309, 400, 405, 428, 429, 432, 433, 435, 437, 438, 451, 452, 466, 467, 468, 469, 470, 474, 475, 476, 477, 478, 479, 485, 486, 490, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 512, 513, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 524, 525, 526, 527, 52

Nu zijn alle geocodes int, dus is het nu opgeschoont.

## stm_prioriteit
Eerst verwijderen we de NaN

In [13]:
df = df.dropna(subset=['stm_prioriteit'])
df['stm_prioriteit']

1         9.0
2         9.0
3         9.0
4         9.0
5         9.0
         ... 
898510    4.0
898516    4.0
898518    5.0
898520    5.0
898522    2.0
Name: stm_prioriteit, Length: 222303, dtype: float64

In [14]:
print(dict(df["stm_prioriteit"].value_counts()))

{5.0: 68078, 4.0: 61530, 2.0: 61381, 9.0: 28574, 8.0: 1767, 1.0: 973}


Hier zien we dat volgens het meegegeven document, in de kolom geen verkeerde data zit

Uit het meegevenen document kunnen we ook vastellen, dat elke rij met stm_prioriteit als 9.0 verwijderd moet worden, omdat hierbij geen aannemer naartoe hoeft.

In [15]:
df = df[~df["stm_prioriteit"].isin([9.0])]

## stm_aanngeb_dd
Eerst verwijderen we de NaN

In [16]:
df = df.dropna(subset=['stm_aanngeb_dd'])
df['stm_aanngeb_dd']

13        03/01/2006
14        03/01/2006
85        10/01/2006
335       01/02/2006
351       05/02/2006
             ...    
898510    11/05/2013
898516    11/05/2013
898518    11/05/2013
898520    11/05/2013
898522    11/05/2013
Name: stm_aanngeb_dd, Length: 193511, dtype: object

Zoals we zien is dit een datetime, hier halen we dus de verkeerde tijden uit

In [17]:
df['stm_aanngeb_dd'] = pd.to_datetime(df['stm_aanngeb_dd'], errors='coerce')

df = df.dropna(subset=['stm_aanngeb_dd'])
df['stm_aanngeb_dd']

13       2006-03-01
14       2006-03-01
85       2006-10-01
335      2006-01-02
351      2006-05-02
            ...    
898510   2013-11-05
898516   2013-11-05
898518   2013-11-05
898520   2013-11-05
898522   2013-11-05
Name: stm_aanngeb_dd, Length: 192956, dtype: datetime64[ns]

Nu hebben we alle verkeerde values eruit gehaald. Nu moeten we de tijden nog omzetten naar een getal die in model in kan. Dit doen met met de Unix-tijdstempel (1970-01-01).  Om dit te doen moeten we van datetime64[ns] naar int gaan en dat delen voor 10**9. Uiteindelijk is dat getal het aantal seconden sinds 1970-01-01 00:00:00.

In [18]:
df['stm_aanngeb_dd'] = df['stm_aanngeb_dd'].astype('int64') / 10**9
df['stm_aanngeb_dd']

13        1.141171e+09
14        1.141171e+09
85        1.159661e+09
335       1.136160e+09
351       1.146528e+09
              ...     
898510    1.383610e+09
898516    1.383610e+09
898518    1.383610e+09
898520    1.383610e+09
898522    1.383610e+09
Name: stm_aanngeb_dd, Length: 192956, dtype: float64

Nu hebben we er int van gemaakt die het aantal seconden sinds 1970-01-01 weergeeft

## stm_oorz_groep
Eerst verwijderen we de NaN

In [19]:
df = df.dropna(subset=['stm_oorz_groep'])
df['stm_oorz_groep']

13        ONR-DERD
14        ONR-DERD
85        ONR-DERD
335       ONR-DERD
351       ONR-DERD
            ...   
898510     TECHONV
898516     TECHONV
898518     TECHONV
898520     TECHONV
898522     TECHONV
Name: stm_oorz_groep, Length: 179929, dtype: object

In [20]:
print(dict(df["stm_oorz_groep"].value_counts()))

{'TECHONV': 117439, 'ONR-DERD': 35295, 'ONR-RIB': 18688, 'WEER': 8507}


Hier zien we dat er geen verkeerde waarde in staan omdat er geen spelfouten of rare waarde in staan, dus is deze kolom te vertrouwen

We willen wel dat de data in deze kolom numeriek is, zodat we dit in het model kunnen gebruiken

In [21]:
df["stm_oorz_groep"] = pd.Categorical(df["stm_oorz_groep"]).codes

## stm_oorz_code
Eerst verwijderen we de NaN

In [22]:
df = df.dropna(subset=['stm_oorz_code'])
df['stm_oorz_code']

13        142.0
14        141.0
85        147.0
335       142.0
351       145.0
          ...  
898510    298.0
898516    221.0
898518    298.0
898520    215.0
898522    218.0
Name: stm_oorz_code, Length: 179929, dtype: float64

In [23]:
print(sorted(set(df["stm_oorz_code"])))

[33.0, 51.0, 130.0, 131.0, 132.0, 133.0, 134.0, 135.0, 136.0, 139.0, 140.0, 141.0, 142.0, 143.0, 144.0, 145.0, 146.0, 147.0, 148.0, 149.0, 150.0, 151.0, 154.0, 180.0, 181.0, 182.0, 183.0, 184.0, 185.0, 186.0, 187.0, 188.0, 189.0, 201.0, 202.0, 203.0, 204.0, 205.0, 206.0, 207.0, 208.0, 209.0, 210.0, 211.0, 212.0, 213.0, 214.0, 215.0, 218.0, 219.0, 220.0, 221.0, 222.0, 223.0, 224.0, 225.0, 226.0, 227.0, 228.0, 229.0, 230.0, 231.0, 233.0, 234.0, 235.0, 239.0, 240.0, 241.0, 242.0, 250.0, 294.0, 298.0, 299.0, 999.0]


Uit het meegegeven document kunnen we vastellen dat de waarde 999 wel in de dataset staan, maar niet in het document. Deze gaan we eerst onderzoeken.

In [24]:
df["stm_oorz_code"].value_counts().loc[999.0]

40

999 komt 40 keer voor, dit is de een duidelijke fout omdat 999 een typsiche waarde is die is ingevult als de stm_oorz_code niet duidelijk is. Deze gaan we nu verwijderen.

In [25]:
df = df[~df["stm_oorz_code"].isin([999.0])]

## stm_contractgeb_gst
Eerst verwijderen we de NaN

In [26]:
df = df.dropna(subset=['stm_contractgeb_gst'])
df['stm_contractgeb_gst']

13         2.0
14         3.0
85        24.0
335        3.0
351       12.0
          ... 
898510    71.0
898516     5.0
898518    71.0
898520     2.0
898522     4.0
Name: stm_contractgeb_gst, Length: 179889, dtype: float64

In [27]:
print(dict(df["stm_contractgeb_gst"].value_counts()))

{4.0: 7689, 5.0: 7525, 9.0: 7426, 12.0: 6500, 30.0: 6263, 8.0: 5214, 2.0: 5200, 3.0: 5106, 31.0: 4988, 34.0: 4935, 32.0: 4931, 26.0: 4897, 11.0: 4734, 71.0: 4632, 7.0: 4400, 10.0: 4289, 13.0: 4246, 53.0: 4153, 24.0: 4045, 19.0: 3774, 29.0: 3705, 37.0: 3601, 21.0: 3574, 27.0: 3508, 36.0: 3442, 20.0: 3427, 18.0: 3344, 25.0: 3311, 35.0: 3213, 6.0: 3210, 81.0: 2947, 51.0: 2883, 28.0: 2809, 62.0: 2764, 23.0: 2743, 52.0: 2424, 1.0: 2229, 61.0: 2157, 14.0: 2104, 22.0: 2023, 58.0: 1803, 54.0: 1595, 63.0: 1485, 59.0: 1466, 55.0: 1287, 15.0: 1261, 33.0: 1142, 64.0: 1135, 60.0: 966, 16.0: 907, 56.0: 729, 17.0: 706, 70.0: 500, 83.0: 218, 50.0: 119, 57.0: 109, 99.0: 57, 82.0: 39}


Hier zien we dat er geen verkeerde waarde in staan omdat er geen spelfouten of rare waarde in staan, dus is deze kolom te vertrouwen. Ook komen alle waardes meerdere keren voor. Dit is een goed teken dat deze waardes kloppen.

## stm_techn_gst
Eerst verwijderen we de NaN

In [28]:
df = df.dropna(subset=['stm_techn_gst'])
df['stm_techn_gst']

13        B
14        B
85        X
335       B
351       S
         ..
898510    S
898516    S
898518    B
898520    B
898522    S
Name: stm_techn_gst, Length: 179889, dtype: object

In [29]:
print(dict(df["stm_techn_gst"].value_counts()))

{'S': 62857, 'B': 45131, 'T': 20463, 'P': 20001, 'E': 14495, 'K': 9404, 'O': 4564, 'G': 958, 'X': 844, 'M': 671, 'I': 457, 'A': 44}


Hier zien we dat er geen verkeerde waarde in staan omdat er geen spelfouten of rare waarde in staan, dus is deze kolom te vertrouwen. Ook komen alle waardes meerdere keren voor. Dit is een goed teken dat deze waardes kloppen.

We willen wel dat de data in deze kolom numeriek is, zodat we dit in het model kunnen gebruiken. Om dit te doen zetten we alle chars om naar hun ASCII waarde.

In [30]:
df["stm_techn_gst"] = df["stm_techn_gst"].apply(lambda x: ord(x))
df["stm_techn_gst"] 

13        66
14        66
85        88
335       66
351       83
          ..
898510    83
898516    83
898518    66
898520    66
898522    83
Name: stm_techn_gst, Length: 179889, dtype: int64

## stm_progfh_in_duur
Eerst verwijderen we de NaN

In [31]:
df = df.dropna(subset=['stm_progfh_in_duur'])
df['stm_progfh_in_duur']

13        99999999.0
14        99999999.0
85        99999999.0
335       99999999.0
351       99999999.0
             ...    
898510            90
898516           180
898518             4
898520            30
898522            52
Name: stm_progfh_in_duur, Length: 179889, dtype: object

De waardes in deze kolom staan in minuten, nu gaan we alle tijden eruit halen die langer zijn dan 8 uur. Dit doen we omdat tijden langer dan 8 uur onnodig zijn om te voorspellen omdat deze tijden onnodig zijn voor reizigers als wij informatie willen leveren voor hoelang het nog duurt todat te treinen weer rijden. Voor dezelfde reden halen we onder de 5 minuten er ook uit. Dit is te kort om als storing gezien te worden.

Ook zien we dat de kolom string als datatype heeft, dit moeten we omzetten naar een numeric datatype zodat we er mee kunnen rekenen.

In [32]:
# Alle waardes in de kolom omzetten naar float
df["stm_progfh_in_duur"] = pd.to_numeric(df["stm_progfh_in_duur"], errors="coerce")

df = df[df["stm_progfh_in_duur"] <= (8 * 60)] # 8 uur
df = df[df["stm_progfh_in_duur"] >= 5] # 5 minuten

## stm_fh_status
Eerst verwijderen we de NaN

In [33]:
df = df.dropna(subset=['stm_fh_status'])
df['stm_fh_status']

139804    4.0
139805    4.0
139810    4.0
139811    1.0
139813    1.0
         ... 
898506    1.0
898510    4.0
898516    4.0
898520    1.0
898522    4.0
Name: stm_fh_status, Length: 116389, dtype: float64

In [34]:
print(dict(df["stm_fh_status"].value_counts()))

{1.0: 75158, 2.0: 19168, 4.0: 17640, 3.0: 3847, 5.0: 576}


Hier zien we dat er geen verkeerde data in zit.

Uit het meegegeven document kunnen we vastellen dat we alle rijen waar de stm_fh_status 4 of 5 is moeten verwijderen. Hier is er namelijk geen functieherstel, dus hoeven wij ook niet te melden hoelang het duurt voordat de treinen weer rijden.

In [35]:
df = df[~df["stm_fh_status"].isin([4.0, 5.0])]

# Target aanmaken
Wat wij willen voorspellen is hoelang het duurt voor de treinen weer rijden, vanaf dat de aannemer zijn prognose heeft geleverd. Om dit te berekenen gebruiken we deze features: stm_progfh_in_datum, stm_progfh_in_tijd, stm_fh_dd, stm_fh_tijd. Hierbij is:     

stm_aanntpl_dd: datum van wanneer de aannemer op locatie is en prognose is aangemaakt.      
stm_aanntpl_tijd: tijd van wanneer de aannemer op locatie is en prognose is aangemaakt.     
stm_fh_dd: datum van wanneer de treinen weer rijden.    
stm_fh_tijd: tijd van wanneer de treinen weer rijden.    

In [36]:
df1 = file[["stm_aanntpl_dd", "stm_aanntpl_tijd", "stm_fh_dd", "stm_fh_tijd"]].dropna()

# Hier zetten we alle datums om naar een datetime en halen we de verkeerde datums eruit met errors='coerce'
df1["stm_aanntpl_dd"] = pd.to_datetime(df1["stm_aanntpl_dd"], format="%d/%m/%Y", errors='coerce')
df1["stm_fh_dd"] = pd.to_datetime(df1["stm_fh_dd"], format="%d/%m/%Y", errors='coerce')
# Dit is het verschil in datums omgezet van dagen naar minuten
datum_verschil = ((df1["stm_fh_dd"] - df1["stm_aanntpl_dd"]).dt.days) * 24 * 60

# Hier zetten we alle tijden om naar een datetime en halen we de verkeerde tijden eruit met errors='coerce'
df1["stm_aanntpl_tijd"] = pd.to_datetime(df1["stm_aanntpl_tijd"], format="%H:%M:%S", errors='coerce')
df1["stm_fh_tijd"] = pd.to_datetime(df1["stm_fh_tijd"], format="%H:%M:%S", errors='coerce')
# Dit is het verschil in tijden omgezet van seconden naar minuten
tijden_verschil = (df1["stm_fh_tijd"] - df1["stm_aanntpl_tijd"]).dt.total_seconds() / 60

# Dit is target, met round() and .astype() maken we er een .0 getal van. Hierdoor komt dit overeen met de format van ander tijd features. 
df['stm_progfh_t_fh'] = round(datum_verschil + tijden_verschil, 0).astype(float)

# Verwijder alle foute tijden die naar NaN zijn omgezet met errors='coerce'
df = df.dropna(subset=["stm_progfh_t_fh"])

Voor dezelfde redenen als bij stm_progfh_in_duur, gaan we alles boven de 8 uur en onder de 5 minuten verwijderen. Dit verwijderd ook alle rijen waar de treinen eerder weer reden dan de prognose was opgeleverd. Dit waren dus niet te vertrouwen rijen.

In [37]:
df = df[df["stm_progfh_t_fh"] <= (8 * 60)] # 8 uur
df = df[df["stm_progfh_t_fh"] >= 5] # 5 minuten


Als laatst sorteren we nog op de meldtijd, hierdoor bestaat onze testset uit de meest recentelijke data, dit geeft dus de meest realistische score voor als we in het echt van nieuwe data de target willen voorspellen. Sorteren doen we op de datum en tijd van wanneer de melding is gemeld.

In [38]:
df = df.sort_values('stm_sap_meld_ddt', ascending=True)
# Index resetten voor de netheid.
df.reset_index()

,index,stm_sap_meld_ddt,stm_geo_mld,stm_prioriteit,stm_aanngeb_dd,stm_oorz_groep,stm_oorz_code,stm_contractgeb_gst,stm_techn_gst,stm_progfh_in_duur,stm_fh_status,stm_progfh_t_fh
0,215640,1.136074e+09,545,4.0,1.136074e+09,2,221.0,19.0,84,55.0,1.0,55.0
1,215641,1.136083e+09,586,2.0,1.136074e+09,0,140.0,9.0,75,40.0,1.0,13.0
2,215647,1.136085e+09,586,2.0,1.136074e+09,2,240.0,9.0,66,46.0,1.0,44.0
3,215644,1.136087e+09,38,2.0,1.136074e+09,0,145.0,30.0,83,12.0,1.0,10.0
4,215648,1.136091e+09,34,2.0,1.136074e+09,0,145.0,30.0,83,9.0,1.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...
90521,894748,1.575643e+09,87,5.0,1.575590e+09,2,231.0,12.0,83,173.0,1.0,64.0
90522,894751,1.575647e+09,510,5.0,1.575590e+09,0,147.0,28.0,83,459.0,1.0,16.0
90523,894756,1.575657e+09,503,5.0,1.575590e+09,0,151.0,31.0,79,390.0,1.0,67.0
90524,894761,1.575666e+09,25,5.0,1.575590e+09,2,213.0,32.0,71,58.0,1.0,19.0


Nu is alle data opgeschoont en is de target aangemaakt. Nu kunnen we alles naar een csv schrijven zodat we dit in een ander notebook kunnen gebruiken.

In [39]:
# Index niet meegeven
df.to_csv('cleaned_data.csv', index=False)